In [1]:
import os
import sys

PYTHON_PATH = r"C:\Program Files\Python310\python.exe"
# Force Spark to use same Python for driver & workers
os.environ["PYSPARK_PYTHON"] = PYTHON_PATH
os.environ["PYSPARK_DRIVER_PYTHON"] = PYTHON_PATH

print("Python executable:", sys.executable)
print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])


Python executable: C:\Program Files\Python312\python.exe
PYSPARK_PYTHON: C:\Program Files\Python310\python.exe


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

spark = SparkSession.builder \
    .appName("FoodNutritionAnalyzerBalanced") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

print("Spark version:", spark.version)


In [ ]:
file_path = r"D:\CDAC PRO\FoodNutritionAnalyzer\FoodNutritionAnalyzer\data\en.openfoodfacts.org.products.csv"

selected_cols = [
    "product_name",
    "nutriscore_grade",
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates-total_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "sodium_100g",
    "additives_n",
    "nova_group"
]

df = spark.read \
    .option("header", "true") \
    .option("sep", "\t") \
    .option("quote", "\u0000") \
    .option("mode", "DROPMALFORMED") \
    .csv(file_path) \
    .select(*selected_cols)

print("Rows loaded:", df.count())

In [ ]:
numeric_cols = [
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates-total_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "sodium_100g",
    "additives_n",
    "nova_group"
]

for c in numeric_cols:
    df = df.withColumn(c, col(c).cast("double"))


In [ ]:
df = df.filter(col("nutriscore_grade").isin("a", "b", "c", "d", "e"))

df = df.withColumn(
    "health_label",
    when(col("nutriscore_grade").isin("a", "b"), 1).otherwise(0)
)

df.groupBy("health_label").count().show()


In [ ]:
healthy_df = df.filter(col("health_label") == 1)
unhealthy_df = df.filter(col("health_label") == 0)

healthy_count = healthy_df.count()
unhealthy_count = unhealthy_df.count()

sampling_ratio = healthy_count / unhealthy_count

unhealthy_sampled = unhealthy_df.sample(
    withReplacement=False,
    fraction=sampling_ratio,
    seed=42
)

balanced_df = healthy_df.union(unhealthy_sampled)

balanced_df.groupBy("health_label").count().show()


In [ ]:
balanced_df = balanced_df.fillna(0)


In [ ]:
feature_cols = [
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates-total_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "sodium_100g",
    "additives_n",
    "nova_group"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

final_df = assembler.transform(balanced_df).select(
    "features", "health_label"
)


In [ ]:
train_df, test_df = final_df.randomSplit([0.8, 0.2], seed=42)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())


In [ ]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="health_label",
    maxIter=50
)

model = lr.fit(train_df)
print("✅ Model trained on balanced data")


In [ ]:
predictions = model.transform(test_df)

evaluator = BinaryClassificationEvaluator(
    labelCol="health_label",
    metricName="areaUnderROC"
)


auc = evaluator.evaluate(predictions)
print("✅ AUC (Balanced Data):", auc)


In [23]:
from pyspark.sql.functions import col

cm_df = predictions.select(
    col("health_label").cast("int").alias("label"),
    col("prediction").cast("int").alias("prediction")
)

cm_df.groupBy("label", "prediction").count().show()


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|         0| 6150|
|    1|         1|63292|
|    0|         1|12591|
|    0|         0|57061|
+-----+----------+-----+



In [24]:
# Check schema
predictions.printSchema()

# Show some predictions
predictions.select(
    "health_label",
    "prediction",
    "probability"
).show(10, truncate=False)


root
 |-- features: vector (nullable = true)
 |-- health_label: integer (nullable = false)
 |-- rawPrediction: vector (nullable = true)
 |-- probability: vector (nullable = true)
 |-- prediction: double (nullable = false)

+------------+----------+----------------------------------------+
|health_label|prediction|probability                             |
+------------+----------+----------------------------------------+
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.02239339469776014,0.9776066053022399]|
|1           |1.0       |[0.022393394697

In [25]:
from pyspark.sql.functions import col

confusion_df = predictions.groupBy(
    col("health_label").alias("label"),
    col("prediction")
).count()

confusion_df.show()


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       1.0|63292|
|    1|       0.0| 6150|
|    0|       0.0|57061|
|    0|       1.0|12591|
+-----+----------+-----+



In [26]:
# Extract counts
tp = predictions.filter("health_label = 1 AND prediction = 1").count()
tn = predictions.filter("health_label = 0 AND prediction = 0").count()
fp = predictions.filter("health_label = 0 AND prediction = 1").count()
fn = predictions.filter("health_label = 1 AND prediction = 0").count()

total = tp + tn + fp + fn

accuracy  = (tp + tn) / total
precision = tp / (tp + fp)
recall    = tp / (tp + fn)
f1_score  = 2 * (precision * recall) / (precision + recall)

print("✅ MODEL TEST RESULTS")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1_score:.4f}")


✅ MODEL TEST RESULTS
Accuracy : 0.8653
Precision: 0.8341
Recall   : 0.9114
F1 Score : 0.8710


In [27]:
# Extract coefficients from Spark Logistic Regression model
import numpy as np
coef = np.array(model.coefficients.toArray())
intercept = model.intercept

print("✅ Coefficients loaded")
print("Coef length:", len(coef))
print("Intercept:", intercept)


✅ Coefficients loaded
Coef length: 11
Intercept: 3.776341309234545


In [28]:
# Feature order MUST match training exactly
features = [
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates-total_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "sodium_100g",
    "additives_n",
    "nova_group"
]

print("✅ Features loaded:", features)


✅ Features loaded: ['energy-kcal_100g', 'fat_100g', 'saturated-fat_100g', 'carbohydrates-total_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g', 'additives_n', 'nova_group']


In [29]:
import numpy as np

def predict_food_health(food_dict):
    """
    Predict health label and confidence using trained Logistic Regression model
    food_dict: nutrition values per 100g
    """

    # Ensure feature order matches training
    X = np.array([food_dict[f] for f in features])

    # Logistic regression equation
    z = np.dot(coef, X) + intercept
    prob_healthy = 1 / (1 + np.exp(-z))

    label = "Healthy" if prob_healthy >= 0.5 else "Unhealthy"
    return label, prob_healthy


In [30]:
sample_food_healthy = {
    "energy-kcal_100g": 120,
    "fat_100g": 3,
    "saturated-fat_100g": 0.5,
    "carbohydrates-total_100g": 18,
    "sugars_100g": 2,
    "fiber_100g": 8,
    "proteins_100g": 10,
    "salt_100g": 0.05,
    "sodium_100g": 0.02,
    "additives_n": 0,
    "nova_group": 1
}

label, confidence = predict_food_health(sample_food_healthy)

print("🥗 HEALTHY SAMPLE TEST")
print("Prediction:", label)
print(f"Healthy confidence: {confidence*100:.2f}%")


🥗 HEALTHY SAMPLE TEST
Prediction: Healthy
Healthy confidence: 92.47%


In [31]:
from pyspark.sql.functions import col

# predictions = model.transform(test_df)

confusion_df = predictions.groupBy(
    col("health_label").alias("Actual"),
    col("prediction").alias("Predicted")
).count()

confusion_df.show()


+------+---------+-----+
|Actual|Predicted|count|
+------+---------+-----+
|     1|      1.0|63292|
|     1|      0.0| 6150|
|     0|      0.0|57061|
|     0|      1.0|12591|
+------+---------+-----+



In [ ]:
import pickle
import numpy as np

class FoodHealthModel:
    def __init__(self, coef, intercept, features):
        self.coef = np.array(coef)
        self.intercept = intercept
        self.features = features

    def predict_proba(self, X):
        z = np.dot(X, self.coef) + self.intercept
        prob_healthy = 1 / (1 + np.exp(-z))
        return np.array([[1 - prob_healthy, prob_healthy]])

    def predict(self, X):
        return (self.predict_proba(X)[0][1] >= 0.5).astype(int)


In [ ]:
model = FoodHealthModel(
    coef=coef,
    intercept=intercept,
    features=features
)

with open("food_health_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("✅ food_health_model.pkl saved")


In [ ]:
print(type(model))


In [1]:
 image_path = r"D:\CDAC PROJECT\FoodNutritionAnalyzer\Screenshot 2026-01-26 152844.png"


text = extract_text_from_image(image_path)
print("📝 OCR TEXT:\n", text)

nutrients = parse_nutrients(text)

nutrients["additives_n"] = infer_additives_n(text)
nutrients["nova_group"] = infer_nova_group(text)

sample_food = {
    "energy-kcal_100g": nutrients["energy_100g"],
    "fat_100g": nutrients["fat_100g"],
    "saturated-fat_100g": nutrients["saturated_fat_100g"],
    "carbohydrates-total_100g": nutrients["carbohydrates_100g"],
    "sugars_100g": nutrients["sugars_100g"],
    "fiber_100g": nutrients.get("fiber_100g", 0.0),
    "proteins_100g": nutrients["proteins_100g"],
    "salt_100g": nutrients["salt_100g"],
    "sodium_100g": nutrients["sodium_100g"] / 1000,
    "additives_n": nutrients["additives_n"],
    "nova_group": nutrients["nova_group"]
}

pred, prob = predict_food_health(model, sample_food)

label = "Healthy" if pred == 1 else "Unhealthy"

print("Prediction:", label)
print("Probability (Unhealthy, Healthy):", prob)


NameError: name 'extract_text_from_image' is not defined

In [43]:
print(sample_food)

{'energy-kcal_100g': 60.0, 'fat_100g': 9.0, 'saturated-fat_100g': 9.0, 'carbohydrates-total_100g': 9.0, 'sugars_100g': 9.0, 'fiber_100g': 0.0, 'proteins_100g': 29.0, 'salt_100g': 0.0, 'sodium_100g': 0.0, 'additives_n': 0, 'nova_group': 4}
